# Per-stage peak comparison

This notebook shows how to compare averaged peaks across distinct time-range **stages** within a single sample (e.g. blank → sample introduction → wash). The workflow is:

1. Load intra-sample timeseries to **visually identify** stage boundaries.
2. Define stages as `(t_min, t_max, name)` tuples.
3. Use `load_peaks_by_stage()` to fetch averaged peaks per stage.
4. Compare peak areas or heights between stages.

> **Tip:** If some peaks are excluded due to missing timeseries data, see the [workaround](#Workaround:-compute-missing-peak-timeseries) at the end of this notebook to trigger their computation.


In [ ]:
from mascope_sdk import MascopeClient

mascope = MascopeClient()

### Pick a sample

List samples and choose one to inspect. Adjust `workspace`, `batches`, and `samples` to match your data.


In [ ]:
# Adjust these to match your data
workspace = "My Workspace"
batch = "My Batch"

sample_list = mascope.samples.list(workspace=workspace, batch=batch)
sample_list

In [ ]:
# Pick a sample by name (or index)
sample_name = sample_list.iloc[0]["sample_item_name"]
sample_id = sample_list.iloc[0]["sample_item_id"]
print(f"Selected sample: {sample_name}")

### Visualise intra-sample timeseries

Load the per-scan timeseries for a compound of interest and plot it. Use this plot to decide where stage boundaries should be.


In [ ]:
# Pick a compound/ion/isotope to plot (adjust to your data)
ion = "NO3-"

ts = mascope.load_peak_timeseries(
    workspace=workspace,
    batches=batch,
    samples=sample_name,
    ion=ion,
)
ts

In [ ]:
import plotly.express as px

fig = px.line(
    ts,
    x="time",
    y="height",
    color="target_isotope_formula",
    title=f"{ion} - timeseries for {sample_name}",
)
fig.update_layout(xaxis_title="Time [s]", yaxis_title="Intensity [cps]")
fig.show()

### Define stages

Based on the plot above, define the time-range stages. Each stage is a tuple of `(t_min, t_max, name)`.


In [ ]:
# Adjust boundaries based on the timeseries plot above
stages = [
    (0, 30, "blank"),
    (30, 120, "sample"),
    (120, 180, "wash"),
]

### Load peaks by stage

`load_peaks_by_stage` fetches the averaged peak list for each stage concurrently and returns a single DataFrame with `stage`, `stage_name`, `t_min`, and `t_max` columns.


In [ ]:
peaks = mascope.load_peaks_by_stage(
    sample=sample_name,
    stages=stages,
)
peaks

### Compare stages


Compare intensities of matched peaks across stages


In [ ]:
# Filter to matched peaks for cleaner comparison
matched = peaks[peaks["target_ion_formula"].notna()].copy()

# Mean height per stage per compound
comparison = (
    matched.groupby(["stage_name", "target_ion_formula"])["height"].mean().reset_index()
)

fig2 = px.bar(
    comparison,
    x="target_ion_formula",
    y="height",
    color="stage_name",
    barmode="group",
    title="Peak height by stage",
)
fig2.update_layout(xaxis_title="Ion", yaxis_title="Mean height")
fig2.show()

Mass defect plot of all peaks per stage


In [ ]:
import numpy as np

df = peaks.sort_values(["mz", "t_min"]).copy()
df["mass_defect"] = df["mz"] - df["mz"].round()
df["log_intensity"] = np.log10(df["area"].clip(lower=1))  # or "height"

fig3 = px.scatter(
    df,
    x="mz",
    y="mass_defect",
    animation_frame="stage_name",
    animation_group="peak_id",
    size="log_intensity",
    size_max=15,
    hover_data=["target_ion_formula", "area"],
    range_x=[df["mz"].min() - 10, df["mz"].max() + 10],
    range_y=[df["mass_defect"].min() - 0.1, df["mass_defect"].max() + 0.1],
    width=900,
    height=700,
    labels={"mz": "m/z", "mass_defect": "Mass defect", "stage": "Stage"},
)
fig3.update_layout(template="plotly_white")
fig3.show()

### Workaround: compute missing peak timeseries

If `load_peaks_by_stage` warns that some peaks were excluded due to missing timeseries, you can trigger their computation by requesting each peak's timeseries individually. The backend computes and caches the result on first request, so subsequent stage-based loads will include all peaks.

**Note:** This may take a while for samples with many peaks.


In [ ]:
# Load the flat peak list for the sample
all_peaks = mascope.samples.get_peaks(sample_id, matches=False)
print(f"Sample has {len(all_peaks)} peaks")

# Request timeseries for each peak (triggers backend computation if missing)
from tqdm import tqdm

i = 0
for _, row in tqdm(
    all_peaks.iterrows(), total=len(all_peaks), desc="Computing timeseries"
):
    mascope.samples.get_peak_timeseries(sample_id, peak_id=row["peak_id"])
    i += 1
    if i > 100:  # Limit to first 50 peaks for demo purposes
        break

print(
    "Done! All timeseries computed. Re-run load_peaks_by_stage() to include all peaks."
)